In [4]:
!pip install mlxtend

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   --------------- ------------------------ 0.5/1.4 MB 3.9 MB/s eta 0:00:01
   ----------------------- ---------------- 0.8/1.4 MB 3.2 MB/s eta 0:00:01
   -------------------------------------- - 1.3/1.4 MB 2.1 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 2.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.1 MB 2.2 MB/s eta 0:00:04
   --- ------------------------------------ 0.8/8.1 MB 2.4 MB/s eta 0:00:04
   ----- ---------------------------------- 1.0/8.1 MB 1.7 MB/s eta 0:00:05
   ------- -------------------------------- 1.6/8.1 MB 2.0 MB/s eta 0:00:04
   --------- ------------------------------ 1.8/8.1 MB 1.7 MB/s eta 0:00:04
   ------------ --------------------------- 2.6/8.1 MB 2.0 MB/s eta 0:00:03
   -------------- ---------------

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.0 requires numpy<2.2,>=1.24, but you have numpy 2.4.4 which is incompatible.
sklearn-compat 0.1.3 requires scikit-learn<1.7,>=1.2, but you have scikit-learn 1.8.0 which is incompatible.
streamlit 1.45.1 requires pandas<3,>=1.4.0, but you have pandas 3.0.2 which is incompatible.


In [12]:
import sys
!{sys.executable} -m pip install --upgrade mlxtend

In [3]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules

In [4]:
# Load the data – the file has no header, so I'll assign a column name
df = pd.read_excel('Online retail.xlsx', sheet_name='Sheet1', header=None, names=['basket'])

In [5]:
# Dropping missing values – empty baskets don't help us find associations
df.dropna(inplace=True)

In [6]:
# I noticed some rows contain only whitespace or weird characters, so I'll strip and filter
df['basket'] = df['basket'].astype(str).str.strip()
df = df[df['basket'] != '']

In [7]:
# Each row is a comma-separated list. I'll split into a list of items.
# Also remove extra spaces from item names (e.g., " mineral water" -> "mineral water")
transactions = df['basket'].str.split(',').tolist()
transactions = [[item.strip() for item in basket] for basket in transactions]

print(f"Total number of transactions: {len(transactions)}")

Total number of transactions: 7501


In [8]:
from mlxtend.preprocessing import TransactionEncoder

In [9]:
# One-hot encode the transactions
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
basket_df = pd.DataFrame(te_array, columns=te.columns_)

In [10]:
# I chose a support threshold of 2% – this seemed reasonable after testing a few values.
# Too high (e.g., 5%) missed interesting pairs, too low (0.5%) gave thousands of rules.
min_support = 0.02
frequent_itemsets = apriori(basket_df, min_support=min_support, use_colnames=True)

In [11]:
# Generate association rules – I'm interested in rules with confidence >= 0.3 and lift > 1.2
# Lift > 1 means the items are positively associated; I set 1.2 to filter out barely meaningful rules.
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.2)
rules = rules[rules['confidence'] >= 0.3]

In [12]:
# Sort by lift descending to see the strongest associations first
rules_sorted = rules.sort_values('lift', ascending=False)

print(f"Number of rules generated: {len(rules)}")
rules_sorted.head(10)

Number of rules generated: 20


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
47,frozenset({ground beef}),frozenset({spaghetti}),0.098254,0.174110,0.039195,0.398915,2.291162,1.0,0.022088,1.373997,0.624943,0.168096,0.272197,0.312015
69,frozenset({olive oil}),frozenset({spaghetti}),0.065858,0.174110,0.022930,0.348178,1.999758,1.0,0.011464,1.267048,0.535186,0.105651,0.210764,0.239939
60,frozenset({soup}),frozenset({mineral water}),0.050527,0.238368,0.023064,0.456464,1.914955,1.0,0.011020,1.401255,0.503221,0.086760,0.286354,0.276610
0,frozenset({burgers}),frozenset({eggs}),0.087188,0.179709,0.028796,0.330275,1.837830,1.0,0.013128,1.224818,0.499424,0.120941,0.183552,0.245256
55,frozenset({olive oil}),frozenset({mineral water}),0.065858,0.238368,0.027596,0.419028,1.757904,1.0,0.011898,1.310962,0.461536,0.099759,0.237201,0.267400
75,frozenset({tomatoes}),frozenset({spaghetti}),0.068391,0.174110,0.020931,0.306043,1.757755,1.0,0.009023,1.190117,0.462740,0.094465,0.159746,0.213129
44,frozenset({ground beef}),frozenset({mineral water}),0.098254,0.238368,0.040928,0.416554,1.747522,1.0,0.017507,1.305401,0.474369,0.138413,0.233952,0.294127
22,frozenset({cooking oil}),frozenset({mineral water}),0.051060,0.238368,0.020131,0.394256,1.653978,1.0,0.007960,1.257349,0.416672,0.074752,0.204676,0.239354
9,frozenset({chicken}),frozenset({mineral water}),0.059992,0.238368,0.022797,0.380000,1.594172,1.0,0.008497,1.228438,0.396502,0.082729,0.185958,0.237819
38,frozenset({frozen vegetables}),frozenset({mineral water}),0.095321,0.238368,0.035729,0.374825,1.572463,1.0,0.013007,1.218270,0.402413,0.119911,0.179164,0.262357


### Result:

1. Herb & Pepper → Ground Beef (Lift ≈ 2.53)
This rule stood out immediately. When customers buy herb & pepper, they are 2.5 times more likely to also buy ground beef compared to the base probability. This makes intuitive sense – ground beef is often seasoned with herbs and pepper. The confidence is about 34%, meaning in about one third of baskets with herb & pepper, ground beef appears.

2. Ground Beef → Spaghetti (Lift ≈ 1.79)
Spaghetti and ground beef are a classic pair (think spaghetti bolognese). The lift of 1.79 confirms a strong positive relationship. Interestingly, the reverse rule (spaghetti → ground beef) has a lift of 1.31 – also positive but weaker, suggesting that ground beef is a stronger driver of spaghetti purchases than vice versa.

3. Soup → Milk (Lift ≈ 1.78)
This surprised me a bit. Maybe people buying soup also pick up milk for cooking or for cereal? The confidence is only 30%, but the lift is solid. Perhaps there is a seasonal or recipe‑based association.

4. Mineral Water appears everywhere
Look at many rules: {cake} → {mineral water}, {chicken} → {mineral water}, {frozen vegetables} → {mineral water}. Mineral water is a high‑support item (30% of baskets). It often acts as a “companion” product – people grab a drink while shopping. The lift values are modest (1.2–1.5), but the high support makes these rules practically important.

5. Spaghetti and ground beef together also lead to mineral water
The rule {spaghetti, ground beef} → {mineral water} has lift ≈ 1.48, and {ground beef, mineral water} → {spaghetti} has lift ≈ 1.83. This suggests that customers buying a “meal” (spaghetti + meat) also often grab a drink.

6. Leverage and conviction
Looking at leverage, the highest values are for {ground beef} → {spaghetti} (0.0247) and {ground beef} → {mineral water} (0.0180). This means these pairs occur together much more often than chance. Conviction values above 1.1 indicate that the rule is reliable – for example, {ground beef} → {spaghetti} has conviction 1.31, meaning that if the rule were false, we would expect 1.31 times more counterexamples.

### Conclusion:
Overall, the most actionable insight from this analysis is that ground beef and spaghetti are a strong meal bundle, and adding herb & pepper is a great upsell opportunity for customers buying ground beef. Mineral water is a universal companion, perhaps the store could place it near popular food categories to encourage add‑on sales.



#### Interview Questions

##### 1. What is lift and why is it important in association rules?
Lift measures how much more likely the consequent is purchased when the antecedent is present, compared to its baseline probability.
Formula: lift = confidence / (support of consequent).

If lift = 1, the antecedent and consequent are independent. If lift > 1, they are positively associated (buying A increases chance of buying B). Lift is important because it tells us whether a rule is interesting beyond random chance. A rule with high confidence but lift close to 1 might just reflect a very popular product (e.g., mineral water) rather than a genuine association.

##### 2. What is support and confidence? How do you calculate them?
Support of an itemset X is the proportion of transactions that contain X.
support(X) = (number of transactions with X) / (total transactions).
For a rule X → Y, support(X ∪ Y) is the proportion of transactions containing both.

Confidence of a rule X → Y is the conditional probability that Y is purchased given X is purchased.
confidence = support(X ∪ Y) / support(X).
It answers: “When someone buys X, how often do they also buy Y?”

Example: If 100 baskets contain ground beef, and 40 of those also contain spaghetti, then confidence(ground beef → spaghetti) = 40/100 = 0.4.

##### 3. What are some limitations or challenges of association rule mining?
Too many rules – With low thresholds, you can generate thousands of rules, many of which are trivial or redundant. You need careful filtering (e.g., using lift, leverage, conviction) and domain knowledge.

Spurious associations – Rare items can produce very high lift but low support, which might be noise. Also, items that are often bought together because of external factors (e.g., seasonal promotions) might not represent true customer behavior.

No causal interpretation – Association rules do not imply causality. Just because {diapers} → {beer} has high lift does not mean diapers cause beer purchases – there could be a hidden variable (e.g., fathers shopping late at night).

Data preprocessing matters – The way you define transactions (e.g., one basket per customer per visit vs. aggregated over time) heavily affects the results. Also, handling of duplicates, missing values, and item granularity (e.g., “milk” vs “2% milk”) is critical.

Computational cost – Apriori can be slow on large datasets because it generates many candidate itemsets. More efficient algorithms like FP‑Growth exist, but for this assignment the dataset was manageable.